# API-Football Data Structure Inspection

## Objective

This notebook closes the **data-structure inspection stage** of the project.

The goal is to understand the structure, granularity, coverage, and variability of the detailed
fixture data collected from API-Football **before designing the analytical database in DuckDB**.

The notebook:

- inspects one real fixture in detail;
- identifies the main nested blocks and their granularities;
- inspects match events, lineups, team statistics, and player statistics;
- validates the observed structure across all collected fixtures;
- checks data availability and empty structures;
- identifies the distinct event and statistic types present in the dataset;
- checks fixture IDs for duplicates or malformed responses;
- summarizes the likely relational entities required by the analytical database.

This notebook is **diagnostic only**. It does not clean, transform, normalize, or write the raw data.


## 1. Setup


In [145]:
import json
from collections import Counter
from pathlib import Path
from pprint import pprint

import pandas as pd


### 1.1 Resolve project paths

The notebook may be executed either from the project root or from the `notebooks/` directory.
The code below resolves the project root automatically.


In [146]:
CWD = Path.cwd()

if (CWD / "pyproject.toml").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "pyproject.toml").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Run the notebook from the project root or the notebooks directory."
    )

DETAILS_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "api_football"
    / "fixtures"
    / "details"
)


In [147]:
if not DETAILS_DIR.exists():
    raise FileNotFoundError(
        f"Detailed fixtures directory not found: {DETAILS_DIR}"
    )

fixture_files = sorted(DETAILS_DIR.glob("fixture_*.json"))

print(f"Detailed fixture files found: {len(fixture_files)}")

if not fixture_files:
    raise FileNotFoundError(
        "No fixture_*.json files were found in the detailed fixtures directory."
    )

for file in fixture_files[:10]:
    print(file.name)


Detailed fixture files found: 1222
fixture_1004052.json
fixture_1004053.json
fixture_1010826.json
fixture_1010827.json
fixture_1016043.json
fixture_1016044.json
fixture_1016055.json
fixture_1016056.json
fixture_1017414.json
fixture_1017415.json


## 2. Load one fixture as a structural example

A single fixture is useful for understanding the nested JSON structure.

However, conclusions drawn from this example will later be validated across **all fixtures**.


In [148]:
def load_fixture_file(path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


sample_file = fixture_files[0]
sample_response = load_fixture_file(sample_file)

print(f"Loaded: {sample_file.name}")
print(f"Response type: {type(sample_response)}")
print(f"Items in response: {len(sample_response) if isinstance(sample_response, list) else 'N/A'}")


Loaded: fixture_1004052.json
Response type: <class 'list'>
Items in response: 1


In [149]:
if not isinstance(sample_response, list) or len(sample_response) != 1:
    raise ValueError(
        "Expected each detailed fixture file to contain a list with exactly one fixture."
    )

fixture = sample_response[0]

print(f"Fixture object type: {type(fixture)}")
print("Top-level keys:")
print(list(fixture.keys()))


Fixture object type: <class 'dict'>
Top-level keys:
['fixture', 'league', 'teams', 'goals', 'score', 'events', 'lineups', 'statistics', 'players']


## 3. Inspect fixture-level blocks

These blocks describe the match itself and are conceptually at **one row per fixture** grain:

- `fixture`;
- `league`;
- `teams`;
- `goals`;
- `score`.


In [150]:
for block in ["fixture", "league", "teams", "goals", "score"]:
    print(f"\n--- {block.upper()} ---")
    pprint(fixture.get(block))



--- FIXTURE ---
{'date': '2023-04-26T19:30:00+00:00',
 'id': 1004052,
 'periods': {'first': 1682537400, 'second': 1682541000},
 'referee': 'Gustavo Correia',
 'status': {'elapsed': 90,
            'extra': None,
            'long': 'Match Finished',
            'short': 'FT'},
 'timestamp': 1682537400,
 'timezone': 'UTC',
 'venue': {'city': 'Vila Nova de Famalicão',
           'id': 1276,
           'name': 'Estádio Municipal 22 de Junho'}}

--- LEAGUE ---
{'country': 'Portugal',
 'flag': 'https://media.api-sports.io/flags/pt.svg',
 'id': 96,
 'logo': 'https://media.api-sports.io/football/leagues/96.png',
 'name': 'Taça de Portugal',
 'round': 'Semi-finals',
 'season': 2022,
 'standings': False}

--- TEAMS ---
{'away': {'id': 212,
          'logo': 'https://media.api-sports.io/football/teams/212.png',
          'name': 'FC Porto',
          'winner': True},
 'home': {'id': 242,
          'logo': 'https://media.api-sports.io/football/teams/242.png',
          'name': 'Famalicao',
     

In [151]:
fixture_id = fixture.get("fixture", {}).get("id")
league_id = fixture.get("league", {}).get("id")
season = fixture.get("league", {}).get("season")
home_team_id = fixture.get("teams", {}).get("home", {}).get("id")
away_team_id = fixture.get("teams", {}).get("away", {}).get("id")

print(f"fixture_id: {fixture_id}")
print(f"league_id: {league_id}")
print(f"season: {season}")
print(f"home_team_id: {home_team_id}")
print(f"away_team_id: {away_team_id}")


fixture_id: 1004052
league_id: 96
season: 2022
home_team_id: 242
away_team_id: 212


### Structural interpretation

The basic match information has **fixture-level granularity**:

`1 fixture -> 1 fixture record`

The fixture contains references to league, season, venue, home team, away team, goals, and score.
These structures can later be normalized into relational entities where useful.


## 4. Inspect team match statistics

`statistics` contains team-level performance statistics for a fixture.

Expected grain:

`1 fixture -> up to N team-statistic blocks`

In normal completed matches, this will typically contain one block for each team.


In [152]:
team_statistics = fixture.get("statistics") or []

print(f"Type: {type(team_statistics)}")
print(f"Team statistic blocks: {len(team_statistics)}")

if team_statistics:
    pprint(team_statistics[0])


Type: <class 'list'>
Team statistic blocks: 2
{'statistics': [{'type': 'Shots on Goal', 'value': 3},
                {'type': 'Shots off Goal', 'value': 4},
                {'type': 'Total Shots', 'value': 8},
                {'type': 'Blocked Shots', 'value': 1},
                {'type': 'Shots insidebox', 'value': 3},
                {'type': 'Shots outsidebox', 'value': 5},
                {'type': 'Fouls', 'value': 21},
                {'type': 'Corner Kicks', 'value': 3},
                {'type': 'Offsides', 'value': 1},
                {'type': 'Ball Possession', 'value': '36%'},
                {'type': 'Yellow Cards', 'value': 3},
                {'type': 'Red Cards', 'value': None},
                {'type': 'Goalkeeper Saves', 'value': 2},
                {'type': 'Total passes', 'value': 297},
                {'type': 'Passes accurate', 'value': 220},
                {'type': 'Passes %', 'value': '74%'},
                {'type': 'expected_goals', 'value': None}],
 'team': {'i

In [153]:
if team_statistics:
    first_team_statistics = team_statistics[0]

    print("Top-level keys:")
    print(list(first_team_statistics.keys()))

    print("\nTeam:")
    pprint(first_team_statistics.get("team"))

    print("\nStatistics:")
    pprint(first_team_statistics.get("statistics"))


Top-level keys:
['team', 'statistics']

Team:
{'id': 242,
 'logo': 'https://media.api-sports.io/football/teams/242.png',
 'name': 'Famalicao'}

Statistics:
[{'type': 'Shots on Goal', 'value': 3},
 {'type': 'Shots off Goal', 'value': 4},
 {'type': 'Total Shots', 'value': 8},
 {'type': 'Blocked Shots', 'value': 1},
 {'type': 'Shots insidebox', 'value': 3},
 {'type': 'Shots outsidebox', 'value': 5},
 {'type': 'Fouls', 'value': 21},
 {'type': 'Corner Kicks', 'value': 3},
 {'type': 'Offsides', 'value': 1},
 {'type': 'Ball Possession', 'value': '36%'},
 {'type': 'Yellow Cards', 'value': 3},
 {'type': 'Red Cards', 'value': None},
 {'type': 'Goalkeeper Saves', 'value': 2},
 {'type': 'Total passes', 'value': 297},
 {'type': 'Passes accurate', 'value': 220},
 {'type': 'Passes %', 'value': '74%'},
 {'type': 'expected_goals', 'value': None}]


In [154]:
sample_stat_types = [
    stat.get("type")
    for team_block in team_statistics
    for stat in (team_block.get("statistics") or [])
    if stat.get("type") is not None
]

print(f"Statistic observations in sample fixture: {len(sample_stat_types)}")
print("Statistic types:")
pprint(sorted(set(sample_stat_types)))


Statistic observations in sample fixture: 34
Statistic types:
['Ball Possession',
 'Blocked Shots',
 'Corner Kicks',
 'Fouls',
 'Goalkeeper Saves',
 'Offsides',
 'Passes %',
 'Passes accurate',
 'Red Cards',
 'Shots insidebox',
 'Shots off Goal',
 'Shots on Goal',
 'Shots outsidebox',
 'Total Shots',
 'Total passes',
 'Yellow Cards',
 'expected_goals']


### Structural interpretation

Team statistics have **fixture-team-statistic granularity**.

A future relational representation can use keys such as:

- `fixture_id`;
- `team_id`;
- `statistic_type`;
- `statistic_value`.

Keeping statistic type/value in a long table is flexible, while a later analytical layer may pivot
selected metrics into columns.


## 5. Inspect match events

`events` represents event-level information occurring during a fixture.

Examples may include:

- goals;
- cards;
- substitutions;
- VAR decisions.

Expected grain:

`1 fixture -> N events`


In [155]:
events = fixture.get("events") or []

print(f"Type: {type(events)}")
print(f"Events found: {len(events)}")

if events:
    pprint(events[0])


Type: <class 'list'>
Events found: 18
{'assist': {'id': 970, 'name': 'Wendell'},
 'comments': None,
 'detail': 'Normal Goal',
 'player': {'id': 773, 'name': 'Iván Marcano'},
 'team': {'id': 212,
          'logo': 'https://media.api-sports.io/football/teams/212.png',
          'name': 'FC Porto'},
 'time': {'elapsed': 16, 'extra': None},
 'type': 'Goal'}


In [156]:
if events:
    first_event = events[0]

    print("Event keys:")
    print(list(first_event.keys()))

    for block in ["time", "team", "player", "assist"]:
        print(f"\n--- {block.upper()} ---")
        pprint(first_event.get(block))


Event keys:
['time', 'team', 'player', 'assist', 'type', 'detail', 'comments']

--- TIME ---
{'elapsed': 16, 'extra': None}

--- TEAM ---
{'id': 212,
 'logo': 'https://media.api-sports.io/football/teams/212.png',
 'name': 'FC Porto'}

--- PLAYER ---
{'id': 773, 'name': 'Iván Marcano'}

--- ASSIST ---
{'id': 970, 'name': 'Wendell'}


In [157]:
sample_event_types = sorted({
    event.get("type")
    for event in events
    if event.get("type") is not None
})

sample_event_details = sorted({
    event.get("detail")
    for event in events
    if event.get("detail") is not None
})

print("Event types in sample fixture:")
pprint(sample_event_types)

print("\nEvent details in sample fixture:")
pprint(sample_event_details)


Event types in sample fixture:
['Card', 'Goal', 'Var', 'subst']

Event details in sample fixture:
['Normal Goal',
 'Red card cancelled',
 'Substitution 1',
 'Substitution 2',
 'Substitution 3',
 'Substitution 4',
 'Substitution 5',
 'Yellow Card']


### Structural interpretation

Events require their own event-level representation because multiple events can occur within the
same fixture and for the same team/player.

Likely identifiers include:

- `fixture_id`;
- event sequence/order generated during transformation;
- `team_id`;
- `player_id`;
- `assist_player_id`;
- elapsed/extra time;
- event type and detail.

The raw API response does not necessarily provide a standalone unique `event_id`, so this should
be handled deliberately when the database is designed.


## 6. Inspect lineups

`lineups` contains team lineup information for a fixture.

It may include:

- team;
- formation;
- coach;
- starting XI;
- substitutes.

The nested players introduce another grain within the lineup.


In [158]:
lineups = fixture.get("lineups") or []

print(f"Type: {type(lineups)}")
print(f"Lineup blocks: {len(lineups)}")

if lineups:
    pprint(lineups[0])


Type: <class 'list'>
Lineup blocks: 2
{'coach': {'id': 2212,
           'name': 'João Pedro Sousa',
           'photo': 'https://media.api-sports.io/football/coachs/2212.png'},
 'formation': '4-2-3-1',
 'startXI': [{'player': {'grid': '1:1',
                         'id': 278619,
                         'name': 'Luíz Júnior',
                         'number': 31,
                         'pos': 'G'}},
             {'player': {'grid': '2:4',
                         'id': 162594,
                         'name': 'Alexandre Penetra',
                         'number': 6,
                         'pos': 'D'}},
             {'player': {'grid': '2:3',
                         'id': 127621,
                         'name': 'Riccieli',
                         'number': 15,
                         'pos': 'D'}},
             {'player': {'grid': '2:2',
                         'id': 26772,
                         'name': 'E. Mihaj',
                         'number': 4,
                    

In [159]:
if lineups:
    first_lineup = lineups[0]

    print("Lineup keys:")
    print(list(first_lineup.keys()))

    print("\nTeam:")
    pprint(first_lineup.get("team"))

    print("\nCoach:")
    pprint(first_lineup.get("coach"))

    print("\nFormation:")
    pprint(first_lineup.get("formation"))

    print("\nStarting XI size:")
    print(len(first_lineup.get("startXI") or []))

    print("\nSubstitutes size:")
    print(len(first_lineup.get("substitutes") or []))


Lineup keys:
['team', 'coach', 'formation', 'startXI', 'substitutes']

Team:
{'colors': {'goalkeeper': {'border': 'ff3300',
                           'number': 'ffffff',
                           'primary': 'ff3300'},
            'player': {'border': 'ffffff',
                       'number': '004080',
                       'primary': 'ffffff'}},
 'id': 242,
 'logo': 'https://media.api-sports.io/football/teams/242.png',
 'name': 'Famalicao'}

Coach:
{'id': 2212,
 'name': 'João Pedro Sousa',
 'photo': 'https://media.api-sports.io/football/coachs/2212.png'}

Formation:
'4-2-3-1'

Starting XI size:
11

Substitutes size:
9


In [160]:
if lineups and (lineups[0].get("startXI") or []):
    print("Example starting-XI record:")
    pprint(lineups[0]["startXI"][0])

if lineups and (lineups[0].get("substitutes") or []):
    print("\nExample substitute record:")
    pprint(lineups[0]["substitutes"][0])


Example starting-XI record:
{'player': {'grid': '1:1',
            'id': 278619,
            'name': 'Luíz Júnior',
            'number': 31,
            'pos': 'G'}}

Example substitute record:
{'player': {'grid': None,
            'id': 301187,
            'name': 'Pablo',
            'number': 77,
            'pos': 'M'}}


### Structural interpretation

Lineups contain at least two useful grains:

- one record per `fixture + team` for formation/coach information;
- one record per `fixture + team + player` for lineup membership and starting/substitute status.

This distinction will matter when the DuckDB schema is created.


## 7. Inspect player match statistics

`players` contains player-level match performance data grouped by team.

Expected grain:

`1 fixture -> N teams -> N players -> N statistic objects`

Player statistics are one of the most nested parts of the detailed response.


In [161]:
player_blocks = fixture.get("players") or []

print(f"Type: {type(player_blocks)}")
print(f"Team player-statistic blocks: {len(player_blocks)}")

if player_blocks:
    pprint(player_blocks[0])


Type: <class 'list'>
Team player-statistic blocks: 2
{'players': [{'player': {'id': 278619,
                         'name': 'Luiz Júnior',
                         'photo': 'https://media.api-sports.io/football/players/278619.png'},
              'statistics': [{'cards': {'red': 0, 'yellow': 0},
                              'dribbles': {'attempts': None,
                                           'past': None,
                                           'success': None},
                              'duels': {'total': None, 'won': None},
                              'fouls': {'committed': None, 'drawn': None},
                              'games': {'captain': False,
                                        'minutes': 90,
                                        'number': 31,
                                        'position': 'G',
                                        'rating': '6.2',
                                        'substitute': False},
                              'goals

In [162]:
if player_blocks:
    first_player_team = player_blocks[0]

    print("Player-team block keys:")
    print(list(first_player_team.keys()))

    print("\nTeam:")
    pprint(first_player_team.get("team"))

    players = first_player_team.get("players") or []
    print(f"\nPlayers in block: {len(players)}")

    if players:
        print("\nExample player record:")
        pprint(players[0])


Player-team block keys:
['team', 'players']

Team:
{'id': 242,
 'logo': 'https://media.api-sports.io/football/teams/242.png',
 'name': 'Famalicao',
 'update': '2023-06-06T04:09:58+00:00'}

Players in block: 20

Example player record:
{'player': {'id': 278619,
            'name': 'Luiz Júnior',
            'photo': 'https://media.api-sports.io/football/players/278619.png'},
 'statistics': [{'cards': {'red': 0, 'yellow': 0},
                 'dribbles': {'attempts': None, 'past': None, 'success': None},
                 'duels': {'total': None, 'won': None},
                 'fouls': {'committed': None, 'drawn': None},
                 'games': {'captain': False,
                           'minutes': 90,
                           'number': 31,
                           'position': 'G',
                           'rating': '6.2',
                           'substitute': False},
                 'goals': {'assists': None,
                           'conceded': 2,
                        

In [163]:
if player_blocks:
    players = player_blocks[0].get("players") or []

    if players:
        first_player = players[0]

        print("Player object:")
        pprint(first_player.get("player"))

        print("\nStatistics object(s):")
        pprint(first_player.get("statistics"))


Player object:
{'id': 278619,
 'name': 'Luiz Júnior',
 'photo': 'https://media.api-sports.io/football/players/278619.png'}

Statistics object(s):
[{'cards': {'red': 0, 'yellow': 0},
  'dribbles': {'attempts': None, 'past': None, 'success': None},
  'duels': {'total': None, 'won': None},
  'fouls': {'committed': None, 'drawn': None},
  'games': {'captain': False,
            'minutes': 90,
            'number': 31,
            'position': 'G',
            'rating': '6.2',
            'substitute': False},
  'goals': {'assists': None, 'conceded': 2, 'saves': 2, 'total': None},
  'offsides': None,
  'passes': {'accuracy': '17', 'key': None, 'total': 28},
  'penalty': {'commited': None,
              'missed': 0,
              'saved': 0,
              'scored': 0,
              'won': None},
  'shots': {'on': None, 'total': None},
  'tackles': {'blocks': None, 'interceptions': None, 'total': None}}]


In [164]:
sample_player_stat_sections = set()

for team_block in player_blocks:
    for player_record in (team_block.get("players") or []):
        for stat_object in (player_record.get("statistics") or []):
            sample_player_stat_sections.update(stat_object.keys())

print("Player statistic sections in sample fixture:")
pprint(sorted(sample_player_stat_sections))


Player statistic sections in sample fixture:
['cards',
 'dribbles',
 'duels',
 'fouls',
 'games',
 'goals',
 'offsides',
 'passes',
 'penalty',
 'shots',
 'tackles']


### Structural interpretation

Player match statistics have **fixture-player** analytical granularity, with nested metric groups
such as games, shots, goals, passes, tackles, duels, dribbles, fouls, cards, and penalties.

During transformation, these nested structures can be flattened into a player-match table.


# 8. Validate the structure across all fixtures

Inspecting a single fixture is not sufficient because API coverage can vary by:

- competition;
- season;
- match status;
- data provider coverage.

The following scan validates the structure across the complete locally collected dataset.

The scan records only structural summaries; it does **not** modify the raw files.


In [165]:
EXPECTED_TOP_LEVEL_KEYS = {
    "fixture",
    "league",
    "teams",
    "goals",
    "score",
    "events",
    "lineups",
    "statistics",
    "players",
}

scan_rows = []

top_level_key_sets = Counter()
missing_top_level_keys = Counter()

event_types = Counter()
event_details = Counter()
team_stat_types = Counter()
player_stat_sections = Counter()
formations = Counter()

malformed_files = []
non_singleton_responses = []
fixture_ids = []

for path in fixture_files:
    try:
        response = load_fixture_file(path)
    except Exception as exc:
        malformed_files.append((path.name, repr(exc)))
        continue

    if not isinstance(response, list) or len(response) != 1:
        non_singleton_responses.append(
            (path.name, type(response).__name__, len(response) if isinstance(response, list) else None)
        )
        continue

    match = response[0]

    if not isinstance(match, dict):
        malformed_files.append((path.name, f"Fixture object is {type(match).__name__}, expected dict"))
        continue

    keys = set(match.keys())
    top_level_key_sets[tuple(sorted(keys))] += 1

    for expected_key in EXPECTED_TOP_LEVEL_KEYS:
        if expected_key not in keys:
            missing_top_level_keys[expected_key] += 1

    fixture_block = match.get("fixture") or {}
    league_block = match.get("league") or {}
    teams_block = match.get("teams") or {}

    match_fixture_id = fixture_block.get("id")
    fixture_ids.append(match_fixture_id)

    match_events = match.get("events") or []
    match_lineups = match.get("lineups") or []
    match_team_statistics = match.get("statistics") or []
    match_player_blocks = match.get("players") or []

    for event in match_events:
        if event.get("type") is not None:
            event_types[event["type"]] += 1
        if event.get("detail") is not None:
            event_details[event["detail"]] += 1

    for team_block in match_team_statistics:
        for stat in (team_block.get("statistics") or []):
            if stat.get("type") is not None:
                team_stat_types[stat["type"]] += 1

    for lineup in match_lineups:
        if lineup.get("formation"):
            formations[lineup["formation"]] += 1

    player_count = 0
    player_stat_object_count = 0

    for team_block in match_player_blocks:
        team_players = team_block.get("players") or []
        player_count += len(team_players)

        for player_record in team_players:
            stat_objects = player_record.get("statistics") or []
            player_stat_object_count += len(stat_objects)

            for stat_object in stat_objects:
                for section_name in stat_object.keys():
                    player_stat_sections[section_name] += 1

    scan_rows.append({
        "file": path.name,
        "fixture_id": match_fixture_id,
        "date": fixture_block.get("date"),
        "status": (fixture_block.get("status") or {}).get("short"),
        "league_id": league_block.get("id"),
        "league_name": league_block.get("name"),
        "season": league_block.get("season"),
        "home_team_id": (teams_block.get("home") or {}).get("id"),
        "away_team_id": (teams_block.get("away") or {}).get("id"),
        "events_count": len(match_events),
        "lineups_count": len(match_lineups),
        "team_statistics_count": len(match_team_statistics),
        "player_team_blocks_count": len(match_player_blocks),
        "players_count": player_count,
        "player_stat_objects_count": player_stat_object_count,
    })

structure_df = pd.DataFrame(scan_rows)

print(f"Files scanned successfully: {len(structure_df)} / {len(fixture_files)}")
print(f"Malformed files: {len(malformed_files)}")
print(f"Non-singleton responses: {len(non_singleton_responses)}")


Files scanned successfully: 1222 / 1222
Malformed files: 0
Non-singleton responses: 0


## 9. Global structural consistency checks


In [166]:
print("Distinct top-level key structures found:")
for key_set, count in top_level_key_sets.most_common():
    print(f"{count:>5} fixtures -> {key_set}")

print("\nExpected top-level keys missing from files:")
if missing_top_level_keys:
    for key, count in missing_top_level_keys.most_common():
        print(f"{key}: {count}")
else:
    print("No expected top-level keys are missing.")


Distinct top-level key structures found:
 1222 fixtures -> ('events', 'fixture', 'goals', 'league', 'lineups', 'players', 'score', 'statistics', 'teams')

Expected top-level keys missing from files:
No expected top-level keys are missing.


In [167]:
if malformed_files:
    print("Malformed files:")
    pprint(malformed_files[:20])

if non_singleton_responses:
    print("\nResponses that did not contain exactly one fixture:")
    pprint(non_singleton_responses[:20])


### 9.1 Fixture ID integrity

Each detailed fixture file should represent one unique fixture.


In [168]:
valid_fixture_ids = [fixture_id for fixture_id in fixture_ids if fixture_id is not None]
fixture_id_counts = Counter(valid_fixture_ids)

duplicate_fixture_ids = {
    fixture_id: count
    for fixture_id, count in fixture_id_counts.items()
    if count > 1
}

print(f"Valid fixture IDs: {len(valid_fixture_ids)}")
print(f"Unique fixture IDs: {len(set(valid_fixture_ids))}")
print(f"Missing fixture IDs: {sum(fixture_id is None for fixture_id in fixture_ids)}")
print(f"Duplicated fixture IDs: {len(duplicate_fixture_ids)}")

if duplicate_fixture_ids:
    pprint(duplicate_fixture_ids)


Valid fixture IDs: 1222
Unique fixture IDs: 1222
Missing fixture IDs: 0
Duplicated fixture IDs: 0


## 10. Data availability and coverage

Empty nested lists are not automatically data errors.

They may indicate:

- a competition for which the API does not provide that level of detail;
- an unfinished/postponed match;
- historical coverage limitations;
- legitimate absence of events.

The objective here is to quantify availability before modelling.


In [169]:
structure_df.head()


,file,fixture_id,date,status,league_id,league_name,season,home_team_id,away_team_id,events_count,lineups_count,team_statistics_count,player_team_blocks_count,players_count,player_stat_objects_count
0,fixture_1004052.json,1004052,2023-04-26T19:30:00+00:00,FT,96,Taça de Portugal,2022,242,212,18,2,2,2,40,40
1,fixture_1004053.json,1004053,2023-05-04T19:30:00+00:00,AET,96,Taça de Portugal,2022,212,242,27,2,2,2,40,40
2,fixture_1010826.json,1010826,2023-03-09T17:45:00+00:00,FT,3,UEFA Europa League,2022,228,42,16,2,2,2,45,45
3,fixture_1010827.json,1010827,2023-03-16T20:00:00+00:00,PEN,3,UEFA Europa League,2022,42,228,29,2,2,2,45,45
4,fixture_1016043.json,1016043,2023-04-13T19:00:00+00:00,FT,3,UEFA Europa League,2022,496,228,14,2,2,2,44,44


In [170]:
coverage_summary = pd.DataFrame({
    "block": [
        "events",
        "lineups",
        "team_statistics",
        "player_statistics",
    ],
    "fixtures_with_data": [
        structure_df["events_count"].gt(0).sum(),
        structure_df["lineups_count"].gt(0).sum(),
        structure_df["team_statistics_count"].gt(0).sum(),
        structure_df["players_count"].gt(0).sum(),
    ],
})

coverage_summary["total_fixtures"] = len(structure_df)
coverage_summary["coverage_pct"] = (
    coverage_summary["fixtures_with_data"]
    / coverage_summary["total_fixtures"]
    * 100
).round(2)

coverage_summary


,block,fixtures_with_data,total_fixtures,coverage_pct
0,events,1194,1222,97.71
1,lineups,1178,1222,96.40
2,team_statistics,1119,1222,91.57
3,player_statistics,1092,1222,89.36


In [171]:
count_columns = [
    "events_count",
    "lineups_count",
    "team_statistics_count",
    "player_team_blocks_count",
    "players_count",
    "player_stat_objects_count",
]

structure_df[count_columns].describe().T


,count,mean,std,min,25%,50%,75%,max
events_count,1222.0,17.360065,4.813438,0.0,15.0,18.0,20.0,36.0
lineups_count,1222.0,1.927987,0.372765,0.0,2.0,2.0,2.0,2.0
team_statistics_count,1222.0,1.831424,0.555866,0.0,2.0,2.0,2.0,2.0
player_team_blocks_count,1222.0,1.787234,0.616907,0.0,2.0,2.0,2.0,2.0
players_count,1222.0,36.165303,12.580246,0.0,40.0,40.0,40.0,52.0
player_stat_objects_count,1222.0,36.165303,12.580246,0.0,40.0,40.0,40.0,52.0


### 10.1 Coverage by competition and season

This is important because availability may not be uniform across the dataset.


In [172]:
coverage_by_competition = (
    structure_df
    .assign(
        has_events=structure_df["events_count"].gt(0),
        has_lineups=structure_df["lineups_count"].gt(0),
        has_team_statistics=structure_df["team_statistics_count"].gt(0),
        has_player_statistics=structure_df["players_count"].gt(0),
    )
    .groupby(
        ["league_id", "league_name", "season"],
        dropna=False,
        as_index=False,
    )
    .agg(
        fixtures=("fixture_id", "count"),
        events_coverage_pct=("has_events", "mean"),
        lineups_coverage_pct=("has_lineups", "mean"),
        team_statistics_coverage_pct=("has_team_statistics", "mean"),
        player_statistics_coverage_pct=("has_player_statistics", "mean"),
    )
)

coverage_pct_columns = [
    "events_coverage_pct",
    "lineups_coverage_pct",
    "team_statistics_coverage_pct",
    "player_statistics_coverage_pct",
]

coverage_by_competition[coverage_pct_columns] = (
    coverage_by_competition[coverage_pct_columns] * 100
).round(2)

coverage_by_competition.sort_values(
    ["season", "league_name"]
).reset_index(drop=True)


,league_id,league_name,season,fixtures,events_coverage_pct,lineups_coverage_pct,team_statistics_coverage_pct,player_statistics_coverage_pct
0,667,Friendlies Clubs,2022,30,43.33,3.33,0.00,0.00
1,94,Primeira Liga,2022,308,100.00,100.00,100.00,100.00
2,550,Super Cup,2022,1,100.00,100.00,0.00,0.00
3,97,Taça da Liga,2022,17,100.00,100.00,47.06,47.06
4,96,Taça de Portugal,2022,17,100.00,100.00,76.47,47.06
5,2,UEFA Champions League,2022,28,100.00,100.00,100.00,100.00
6,848,UEFA Europa Conference League,2022,2,100.00,100.00,100.00,100.00
7,3,UEFA Europa League,2022,12,100.00,100.00,100.00,100.00
8,667,Friendlies Clubs,2023,24,79.17,70.83,50.00,8.33
9,94,Primeira Liga,2023,308,100.00,100.00,100.00,100.00


### 10.2 Match-status distribution

Some missing data can be explained by fixture status, so status should be inspected before
interpreting empty statistics as missing coverage.


In [173]:
structure_df["status"].value_counts(dropna=False).rename_axis("status").to_frame("fixtures")


,fixtures
status,
FT,1200
PEN,10
AET,7
CANC,4
Canc,1


## 11. Distinct event types and event details

These values are collected from the **entire dataset**, not only the sample fixture.


In [174]:
event_types_df = pd.DataFrame(
    event_types.most_common(),
    columns=["event_type", "observations"],
)

event_types_df


,event_type,observations
0,subst,11055
1,Card,6324
2,Goal,3376
3,Var,459


In [175]:
event_details_df = pd.DataFrame(
    event_details.most_common(),
    columns=["event_detail", "observations"],
)

event_details_df


,event_detail,observations
0,Yellow Card,5977
1,Normal Goal,2830
2,Substitution 1,2352
3,Substitution 2,2347
4,Substitution 3,2325
5,Substitution 4,2171
6,Substitution 5,1664
7,Penalty,407
8,Red Card,347
9,Penalty confirmed,136


## 12. Distinct team statistic types


In [176]:
team_stat_types_df = pd.DataFrame(
    team_stat_types.most_common(),
    columns=["statistic_type", "observations"],
)

team_stat_types_df


,statistic_type,observations
0,Shots on Goal,2238
1,Shots off Goal,2238
2,Total Shots,2238
3,Blocked Shots,2238
4,Shots insidebox,2238
5,Shots outsidebox,2238
6,Fouls,2238
7,Corner Kicks,2238
8,Offsides,2238
9,Ball Possession,2238


## 13. Player statistic sections

Player match statistics are nested into groups. This scan identifies which top-level groups
actually occur across the complete dataset.


In [177]:
player_stat_sections_df = pd.DataFrame(
    player_stat_sections.most_common(),
    columns=["statistic_section", "observations"],
)

player_stat_sections_df


,statistic_section,observations
0,games,44194
1,offsides,44194
2,shots,44194
3,goals,44194
4,passes,44194
5,tackles,44194
6,duels,44194
7,dribbles,44194
8,fouls,44194
9,cards,44194


## 14. Formation coverage

This is not required for the database design itself, but it confirms the values available in
lineup data and helps reveal whether lineup coverage is meaningful.


In [178]:
formations_df = pd.DataFrame(
    formations.most_common(),
    columns=["formation", "observations"],
)

formations_df


,formation,observations
0,4-2-3-1,862
1,3-4-3,432
2,4-3-3,311
3,3-4-2-1,210
4,4-4-2,182
5,4-1-4-1,84
6,5-4-1,59
7,3-5-2,57
8,5-3-2,29
9,3-4-1-2,20


# 15. Identify dataset granularities

Based on the API structure and the global validation, the main analytical grains are:

| Data block | Natural grain | Typical relationship to fixture |
|---|---|---|
| Fixture | fixture | 1 |
| League / season reference | league + season | referenced by many fixtures |
| Team | team | referenced by many fixtures |
| Fixture-team | fixture + team | 2 per normal match |
| Team match statistic | fixture + team + statistic type | many |
| Event | fixture + event occurrence | many |
| Lineup team | fixture + team | up to 2 |
| Lineup player | fixture + team + player | many |
| Player match statistics | fixture + player | many |

This is the main output required before designing the relational model.


## 16. Candidate relational entities

The following is a **schema proposal**, not the final database implementation.

The next stage should decide which entities become physical DuckDB tables and how raw nested
structures will be flattened.


In [179]:
schema_candidates = pd.DataFrame([
    {
        "entity": "fixtures",
        "grain": "one row per fixture",
        "primary_key_candidate": "fixture_id",
        "source_block": "fixture + league + teams + goals + score",
    },
    {
        "entity": "teams",
        "grain": "one row per team",
        "primary_key_candidate": "team_id",
        "source_block": "teams / team references",
    },
    {
        "entity": "competitions",
        "grain": "one row per competition",
        "primary_key_candidate": "league_id",
        "source_block": "league",
    },
    {
        "entity": "team_match_statistics",
        "grain": "one row per fixture + team + statistic type",
        "primary_key_candidate": "composite key",
        "source_block": "statistics",
    },
    {
        "entity": "events",
        "grain": "one row per event occurrence",
        "primary_key_candidate": "generated event key / composite ordering",
        "source_block": "events",
    },
    {
        "entity": "lineups",
        "grain": "one row per fixture + team",
        "primary_key_candidate": "fixture_id + team_id",
        "source_block": "lineups",
    },
    {
        "entity": "lineup_players",
        "grain": "one row per fixture + team + player",
        "primary_key_candidate": "composite key",
        "source_block": "lineups.startXI / substitutes",
    },
    {
        "entity": "player_match_statistics",
        "grain": "one row per fixture + player",
        "primary_key_candidate": "fixture_id + player_id",
        "source_block": "players",
    },
])

schema_candidates


,entity,grain,primary_key_candidate,source_block
0,fixtures,one row per fixture,fixture_id,fixture + league + teams + goals + score
1,teams,one row per team,team_id,teams / team references
2,competitions,one row per competition,league_id,league
3,team_match_statistics,one row per fixture + team + statistic type,composite key,statistics
4,events,one row per event occurrence,generated event key / composite ordering,events
5,lineups,one row per fixture + team,fixture_id + team_id,lineups
6,lineup_players,one row per fixture + team + player,composite key,lineups.startXI / substitutes
7,player_match_statistics,one row per fixture + player,fixture_id + player_id,players


# 17. Final diagnostic summary

Run this cell after all previous sections.

It summarizes whether the raw dataset is structurally ready for the next stage.


In [180]:
print("DATA STRUCTURE INSPECTION SUMMARY")
print("=" * 40)
print(f"Fixture files discovered: {len(fixture_files)}")
print(f"Fixtures scanned successfully: {len(structure_df)}")
print(f"Unique valid fixture IDs: {len(set(valid_fixture_ids))}")
print(f"Malformed files: {len(malformed_files)}")
print(f"Non-singleton responses: {len(non_singleton_responses)}")
print(f"Duplicate fixture IDs: {len(duplicate_fixture_ids)}")
print(f"Distinct top-level structures: {len(top_level_key_sets)}")
print(f"Distinct event types: {len(event_types)}")
print(f"Distinct event details: {len(event_details)}")
print(f"Distinct team statistic types: {len(team_stat_types)}")
print(f"Distinct player statistic sections: {len(player_stat_sections)}")

print("\nCoverage:")
display(coverage_summary)


DATA STRUCTURE INSPECTION SUMMARY
Fixture files discovered: 1222
Fixtures scanned successfully: 1222
Unique valid fixture IDs: 1222
Malformed files: 0
Non-singleton responses: 0
Duplicate fixture IDs: 0
Distinct top-level structures: 1
Distinct event types: 4
Distinct event details: 27
Distinct team statistic types: 18
Distinct player statistic sections: 11

Coverage:


,block,fixtures_with_data,total_fixtures,coverage_pct
0,events,1194,1222,97.71
1,lineups,1178,1222,96.40
2,team_statistics,1119,1222,91.57
3,player_statistics,1092,1222,89.36
